# DwellIQ — Exploratory Data Analysis & Model Prototyping 🏡

**Project:** DwellIQ — Smarter Estimates. Better Decisions.  
**Academic Title:** House Price Prediction Using Linear Regression and Random Forest  
**Author:** Vaishnavi Naik  

This interactive notebook demonstrates the end-to-end data science lifecycle:
1. **Data Ingestion & Cleaning:** Parsing textual denominations (Cr & Lac) into absolute Indian Rupees (₹).
2. **Exploratory Data Analysis (EDA):** Statistical distributions, geographical listing volume, and bedroom counts.
3. **Pipeline Construction:** Preprocessing using `ColumnTransformer` with median/frequent imputation and `OneHotEncoder`.
4. **Model Comparison:** Evaluating Linear Regression vs. Random Forest Regressor on held-out test splits.

In [ ]:
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set project directory
PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "house_prices.csv"

print(f"Loading raw dataset from: {DATA_PATH}")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Raw dataset dimensions: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)

## 1. Target Variable Parsing (Indian Currency Units)
Listing prices are recorded as string units like `'42 Lac'` or `'1.4 Cr'`. We standardize these into continuous values.

In [ ]:
def convert_price(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower().replace(",", "").replace("₹", "").replace("inr", "").strip()
    match = re.fullmatch(r"(\d+(?:\.\d+)?)\s*(crores?|cr|lakhs?|lacs?)", text)
    if match:
        amount = float(match.group(1))
        unit = match.group(2)
        return amount * 10_000_000 if unit in {"cr", "crore", "crores"} else amount * 100_000
    return np.nan

df_raw["total_price_rupees"] = df_raw["Amount(in rupees)"].apply(convert_price)
df_valid = df_raw.dropna(subset=["total_price_rupees"]).copy()
print(f"Valid price listings: {len(df_valid):,}")
df_valid[["Amount(in rupees)", "total_price_rupees"]].head()

## 2. Geographical Distribution & Pricing
Analyze listing frequency across top Indian real estate markets.

In [ ]:
top_locations = df_valid["location"].value_counts().head(10)
plt.figure(figsize=(10, 4.5))
top_locations.plot(kind="bar", color="#2D5A43", edgecolor="#18261E")
plt.title("Top 10 Cities by Listing Volume", fontsize=13, fontweight="bold")
plt.ylabel("Number of Listings")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()

## 3. Pipeline Model Evaluation & Feature Importance
Load the saved production pipeline and inspect the top predictive feature associations.

In [ ]:
import joblib

MODEL_PATH = PROJECT_ROOT / "models" / "best_model.joblib"
bundle = joblib.load(MODEL_PATH)
pipeline = bundle["model"]
metrics = bundle.get("metrics", {})

print(f"Selected Model: {bundle.get('model_name')}")
print(f"Mean Absolute Error: ₹{float(metrics.get('MAE_Rupees', 0)):,.0f}")
print(f"R² Score: {float(metrics.get('R2', 0)):.4f}")

# Extract Feature Importances
rf = pipeline.named_steps["model"]
prep = pipeline.named_steps["preprocessor"]
feat_names = prep.get_feature_names_out()
top_idx = np.argsort(rf.feature_importances_)[::-1][:8]

print("\nTop 8 Predictive Feature Signals:")
for i in top_idx:
    print(f"- {feat_names[i]}: {rf.feature_importances_[i]:.4f}")